# L00 · 15분에 보는 LLM RL 전체 지도

## Goal

- agent·환경·reward를 구분한다
- PPO·DPO·GRPO·Agentic RL의 위치를 찾는다
- 좋은 action 확률의 변화를 읽는다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L00:toy:42").hexdigest()
print(f"lesson=L00 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L00 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:3bd2b5bff2836ea5d5c5b1cb21352bdd5850e9e5a4c9d2fe6c83b08e2e7cebb4 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: **전체 지도** → 확률·최적화 → bandit/MDP → policy gradient/PPO → LLM 정렬 → Agentic RL → 평가

$$J(\theta)=\mathbb{E}_{a\sim\pi_\theta}[r(a)]$$

RL은 agent가 관측을 보고 action을 고르고, 환경의 feedback으로 미래 action 분포를 바꾸는 학습입니다. LLM에서는 action이 token 또는 tool call이고, DPO는 저장된 선호쌍으로 이 효과를 직접 최적화하며, PPO·GRPO는 새 응답을 rollout해 reward를 받습니다.

```mermaid
flowchart LR
  P[확률·최적화] --> B[Bandit·MDP]
  B --> PG[Policy gradient·PPO]
  PG --> L[LLM policy·reward]
  L --> R[RLHF·DPO·GRPO·DAPO]
  L --> A[Agentic RL]
  R --> E[평가·재현]
  A --> E
```

Mermaid가 보이지 않을 때의 동등한 지도:

```text
확률·최적화 → bandit → MDP/Bellman → MC/TD/Q-learning → DQN
             └→ policy gradient → actor-critic/GAE → PPO
                                      └→ LLM policy + preference/reward
                                           ├→ RLHF-PPO
                                           ├→ DPO
                                           ├→ GRPO/RLVR → DAPO
                                           └→ Agentic RL
모든 경로 → 평가·reward hacking 진단·재현성
```

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** reward=1인 action의 확률은 20회 update 뒤 어느 방향으로 갈까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>목적함수의 부호가 맞다면 증가합니다. 이것이 뒤의 모든 알고리즘에서 확인할 최소 단위입니다.</details>

In [2]:
logits = torch.zeros(2, requires_grad=True)
optimizer = torch.optim.SGD([logits], lr=0.4)
reward_by_action = torch.tensor([0.0, 1.0])
probability_history = []
for _ in range(20):
    probabilities = torch.softmax(logits, dim=-1)
    loss = -(probabilities * reward_by_action).sum()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    probability_history.append(float(probabilities[1].detach()))
print({"p_good_start": round(probability_history[0], 3),
       "p_good_end": round(probability_history[-1], 3)})

{'p_good_start': 0.5, 'p_good_end': 0.916}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 두 action으로 시작하면 상태·credit assignment를 잠시 치워 두고 `확률 → reward → gradient` 고리만 볼 수 있습니다. 대안인 큰 trainer는 현실적이지만 첫 오류의 원인을 분리하기 어렵습니다.

**흔한 함정:** loss에 음수를 빠뜨리면 optimizer가 좋은 action을 줄입니다. `p_good_end > p_good_start` 회귀 검사가 부호 오류를 잡습니다. 회귀 test: `test_reinforce_sign`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert probability_history[-1] > probability_history[0] > 0.0
print("checks=passed")

checks=passed


**회상 문제:** PPO·DPO·GRPO 중 새 응답을 rollout하지 않아도 되는 것은 무엇이며, 왜 그런가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 출력에서 좋은 action 확률이 0.5에서 0.916으로 증가했습니다. 이는 이 toy 목적함수의 방향만 검증하며 알고리즘 간 우열을 뜻하지 않습니다.
- 실제 확인: `test_reinforce_sign`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L01에서 이 확률 변화의 재료인 log-probability, entropy, KL과 gradient를 직접 계산합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `dpo-2023` — `docs/sources.yml`
- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `agent-lightning-2025` — `docs/sources.yml`